# RTE student: DeBERTa-v3-xsmall, plain and distilled

Same code and pinned packages as the local venv and the teacher notebook. Produces:
1. `outputs/student_deberta_v3_xsmall/`: student trained on hard labels only (no-distillation comparison)
2. `outputs/distill_deberta_v3_xsmall/`: student distilled from the DeBERTa-v3-base teacher (the submission)

**Distillation needs the teacher logits** from `colab_teacher.ipynb`:
- Same runtime as the teacher notebook: they're already at `outputs/baseline_deberta_v3_base_oof/teacher_logits_train.npy`.
- New runtime: upload `teacher_artifacts.zip` in step 5.

The plain student doesn't need the teacher, so it can run before the teacher finishes.

Runtime → Change runtime type → **GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import sys; print(sys.version)

## 1. Get the code
Skips the clone if the repo is already here (e.g. same runtime as the teacher notebook).

In [ ]:
import os
REPO_URL = "https://github.com/ihemanthm/rte-challenge.git"  # private repo: upload a zip instead

%cd /content
if not os.path.isdir("/content/rte-challenge"):
    if REPO_URL:
        !git clone -q $REPO_URL rte-challenge
    else:
        from google.colab import files
        up = files.upload()  # choose rte-challenge.zip
        !unzip -q -o {next(iter(up))} -d /content
%cd /content/rte-challenge
!git log --oneline -1 2>/dev/null || true

## 2. Install pinned dependencies (keep Colab's CUDA torch)

In [ ]:
!grep -vE '^(torch|nvidia-|triton|cuda-)' requirements.txt > /tmp/req-colab.txt
!pip install -q -r /tmp/req-colab.txt
!pip install -q --no-deps -e .
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())

## 3. Data
Skipped if `data/` already exists; otherwise upload the challenge zip.

In [ ]:
%cd /content/rte-challenge
if not (os.path.exists("data/train.jsonl") and os.path.exists("data/dev.jsonl")):
    from google.colab import files
    up = files.upload()  # choose the challenge dataset zip
    ZIP = next(iter(up))
    !python scripts/prepare_data.py --zip "$ZIP" --out data
!wc -l data/train.jsonl data/dev.jsonl

## 4. Plain student: hard labels only, 5 seeds

In [ ]:
!python -m rte.train --config configs/student_deberta_v3_xsmall.yaml

## 5. Teacher logits
Uses the logits from the teacher run if they're on this machine; otherwise upload `teacher_artifacts.zip`.

In [ ]:
LOGITS = "outputs/baseline_deberta_v3_base_oof/teacher_logits_train.npy"
if not os.path.exists(LOGITS):
    from google.colab import files
    up = files.upload()  # choose teacher_artifacts.zip from colab_teacher.ipynb
    !unzip -q -o {next(iter(up))} -d /content/rte-challenge
import numpy as np
print(LOGITS, np.load(LOGITS).shape)  # expect (2490, 2)

## 6. Distilled student: the submission, 5 seeds
Same hyperparameters as step 4; only the loss changes (alpha=0.5 CE + 0.5 KD at T=2).

In [ ]:
!python -m rte.distill --config configs/distill_deberta_v3_xsmall.yaml

## 7. Compare and download
Unzip `student_artifacts.zip` in the local project root. The distilled checkpoint is kept (~280 MB) so you can re-run predictions locally; the plain student's is dropped.

In [ ]:
import json, glob
for f in sorted(glob.glob("outputs/*/metrics.json")):
    m = json.load(open(f))
    p = m["params"]
    print(f"{m['run_name']:32s} dev {m['dev_acc_mean']*100:5.1f} ± {m['dev_acc_std']*100:4.1f}  "
          f"params {p['total']/1e6:6.1f}M (non-emb {p['non_embedding']/1e6:5.1f}M)")

!zip -q -r student_artifacts.zip outputs/student_deberta_v3_xsmall outputs/distill_deberta_v3_xsmall -x 'outputs/student_deberta_v3_xsmall/model/*'
!du -h student_artifacts.zip
from google.colab import files
files.download("student_artifacts.zip")